In [34]:
import numpy as np
import pandas as pd

In [35]:
pd.set_option('display.max_columns', None)

In [36]:
dataset_workout = pd.read_csv('../../datas/dataset_workout.csv')

In [37]:
df = pd.read_csv('../../datas/dataset_users.csv')
df_weekly_progress = pd.read_csv('../../datas/dataset_userprogress.csv')

df_merged = pd.merge(
    df,                       # Tabel kiri (Data Profil)
    df_weekly_progress,       # Tabel kanan (Data Mingguan)
    on='User_ID',             # Kunci Penghubung
    how='left'                # Jenis Join
)

In [38]:
MUSCLE_GROUPS = {
    'Chest': ['pectorals', 'chest', 'serratus anterior'],
    'Back': ['lats', 'latissimus dorsi', 'trapezius', 'upper back', 'lower back', 'spine'],
    'Shoulders': ['delts', 'deltoids', 'shoulders', 'rotator cuff'],
    'Triceps': ['triceps'],
    'Biceps': ['biceps', 'brachialis'],
    'Legs': ['quads', 'quadriceps', 'hamstrings', 'glutes', 'calves', 'abductors', 'adductors'],
    'Abs': ['abs', 'abdominals', 'obliques']
}

In [39]:
def clean_string_data(text):
    """
    Fungsi untuk mengubah "['cable']" menjadi "cable"
    Sangat berguna karena CSV membaca list sebagai teks biasa.
    """
    if pd.isna(text): 
        return "None"
    # Menghapus kurung siku dan tanda kutip satu per satu
    cleaned = str(text).replace("[", "").replace("]", "").replace("'", "")
    return cleaned.strip()

In [40]:
def get_exercises_by_muscle(df_exercises, target_groups, limit=3):
    keywords = []
    for group in target_groups:
        keywords.extend(MUSCLE_GROUPS.get(group, []))
    
    # Pencarian tetap jalan karena "calves" ada di dalam "['calves']"
    mask = df_exercises['targetMuscles'].astype(str).apply(
        lambda x: any(k.lower() in x.lower() for k in keywords)
    )
    
    df_filtered = df_exercises[mask]
    
    if len(df_filtered) > 0:
        return df_filtered.sample(n=min(limit, len(df_filtered)), replace=False)
    else:
        return pd.DataFrame()

In [41]:
def generate_weekly_plan(user_row, df_exercises):
    user_id = user_row['User_ID']
    freq = user_row['Workout_Frequency_x'] 
    duration = user_row['Average_Duration_Minutes_x']
    goal = user_row['Goal_x']
    
    total_exercises_per_day = max(4, int(duration / 10))
    weekly_schedule = []
    
    # --- TENTUKAN TEMA HARI BERDASARKAN FREKUENSI ---
    # Kita pakai IF ELIF supaya tidak kena KeyError lagi
    schedule_map = {}
    
    if freq <= 2:
        schedule_map = {
            1: {'Theme': 'Full Body A', 'Focus': ['Legs', 'Chest', 'Shoulders', 'Triceps']},
            2: {'Theme': 'Full Body B', 'Focus': ['Back', 'Legs', 'Biceps', 'Abs']}
        }
    elif freq == 3:
        schedule_map = {
            1: {'Theme': 'Push Day', 'Focus': ['Chest', 'Shoulders', 'Triceps']},
            2: {'Theme': 'Pull Day', 'Focus': ['Back', 'Biceps', 'Abs']},
            3: {'Theme': 'Leg Day', 'Focus': ['Legs']}
        }
    elif freq == 4:
        schedule_map = {
            1: {'Theme': 'Upper A', 'Focus': ['Chest', 'Back', 'Shoulders']},
            2: {'Theme': 'Lower A', 'Focus': ['Legs', 'Abs']},
            3: {'Theme': 'Upper B', 'Focus': ['Biceps', 'Triceps', 'Chest']},
            4: {'Theme': 'Lower B', 'Focus': ['Legs', 'Abs']}
        }
    else: # Untuk freq 5, 6, atau 7
        schedule_map = {
            1: {'Theme': 'Push', 'Focus': ['Chest', 'Shoulders', 'Triceps']},
            2: {'Theme': 'Pull', 'Focus': ['Back', 'Biceps']},
            3: {'Theme': 'Legs', 'Focus': ['Legs', 'Abs']},
            4: {'Theme': 'Upper Body', 'Focus': ['Chest', 'Back', 'Arms']},
            5: {'Theme': 'Lower Body', 'Focus': ['Legs', 'Abs']},
            6: {'Theme': 'Cardio/Abs', 'Focus': ['Abs', 'Cardio']}
        }

    # --- LOOPING MEMBUAT JADWAL ---
    for day_num, config in schedule_map.items():
        # Berhenti jika hari sudah melebihi frekuensi yang diminta user
        if day_num > freq: 
            break
            
        theme = config['Theme']
        muscles = config['Focus']
        slot_per_muscle = max(1, total_exercises_per_day // len(muscles))
        
        for muscle in muscles:
            exercises = get_exercises_by_muscle(df_exercises, [muscle], limit=slot_per_muscle)
            
            for _, ex in exercises.iterrows():
                # Tentukan Sets & Reps
                if goal == 'Muscle Gain':
                    sets, reps = 3, "8-12"
                elif goal == 'Weight Loss':
                    sets, reps = 4, "12-15"
                else:
                    sets, reps = 3, "10-12"
                
                weekly_schedule.append({
                    'User_ID': user_id,
                    'Day': f"Day {day_num} - {theme}",
                    'Muscle Group': muscle,
                    'Exercise Name': ex['name'],
                    'Equipment': clean_string_data(ex['equipments']),
                    'Sets': sets,
                    'Reps': reps,
                    'Instructions': clean_string_data(ex['instructions'])
                })
                
    return pd.DataFrame(weekly_schedule)

In [43]:
all_user_plans = []
unique_users_df = df_merged.drop_duplicates(subset=['User_ID'])

for index, user_row in unique_users_df.iterrows():
    my_plan = generate_weekly_plan(user_row, dataset_workout)
    if not my_plan.empty:
        all_user_plans.append(my_plan)

# Gabung semua jadwal jadi satu tabel panjang
df_all_plans = pd.concat(all_user_plans, ignore_index=True)

In [44]:
df_final = pd.merge(
    df_merged,      # Data Mingguan (Week 0-12)
    df_all_plans,   # Data Latihan (Day 1-X)
    on='User_ID',   # Disambung pake User ID
    how='left'      # Left Join
)

print(f"Sukses! df_final berhasil dibuat dengan ukuran: {df_final.shape}")

Sukses! df_final berhasil dibuat dengan ukuran: (21853, 61)


In [45]:
df_final.head()

,User_ID,Age_x,Gender_x,Height_cm_x,Initial_Weight_kg_x,Initial_BMI_x,BMI_Category,Body_Fat_Category_x,Body_Fat_Percentage,Goal_x,Workout_Frequency_x,Average_Duration_Minutes_x,level_x,Badminton_x,Football_x,Basketball_x,Tennis_x,Volleyball_x,Table_Tennis_x,Swim_x,Age_y,Gender_y,Height_cm_y,Initial_Weight_kg_y,Initial_BMI_y,BMI_Category_x,Body_Fat_Category_y,Body_Fat_Percentage_x,Goal_y,Workout_Frequency_y,Average_Duration_Minutes_y,level_y,Badminton_y,Football_y,Basketball_y,Tennis_y,Volleyball_y,Table_Tennis_y,Swim_y,Week,Weight_kg,BMI,Body_Fat_Percentage_y,Daily_Calories,Daily_Water_ml,Target_Protein_g,Target_Carbs_g,Target_Fat_g,Limit_Sugar_g,Target_Fiber_g,Limit_Cholesterol_mg,Target_Calcium_mg,Meal_Frequency,BMI_Category_y,Day,Muscle Group,Exercise Name,Equipment,Sets,Reps,Instructions
0,1,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,0,53.0,16.18,10.9,2281,2755,171,285,50,57,31,300,1000,5,Underweight,Day 1 - Full Body A,Legs,bent knee lying twist (male),body weight,3,8-12,Step:1 Lie flat on your back with your knees b...
1,1,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,0,53.0,16.18,10.9,2281,2755,171,285,50,57,31,300,1000,5,Underweight,Day 1 - Full Body A,Legs,barbell standing rocking leg calf raise,barbell,3,8-12,Step:1 Stand with your feet shoulder-width apa...
2,1,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,0,53.0,16.18,10.9,2281,2755,171,285,50,57,31,300,1000,5,Underweight,Day 1 - Full Body A,Chest,cable decline fly,cable,3,8-12,Step:1 Adjust the cable machine to a decline p...
3,1,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,0,53.0,16.18,10.9,2281,2755,171,285,50,57,31,300,1000,5,Underweight,Day 1 - Full Body A,Chest,smith incline bench press,smith machine,3,8-12,Step:1 Adjust the bench to a 30-45 degree incl...
4,1,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,45,Male,181,53,16.18,Underweight,2.0,10.9,Muscle Gain,2,90,Beginner,0,1,0,0,1,1,0,0,53.0,16.18,10.9,2281,2755,171,285,50,57,31,300,1000,5,Underweight,Day 1 - Full Body A,Shoulders,dumbbell one arm upright row,dumbbell,3,8-12,Step:1 Stand with your feet shoulder-width apa...


In [47]:
# List ID yang mau dilihat
target_ids = [5]

# Filter df_final
result_df = df_final[df_final['User_ID'].isin(target_ids)]

# Urutkan biar rapi
result_df = result_df.sort_values(by=['User_ID', 'Week', 'Day'])

# Tampilkan kolom penting saja
cols_view = ['User_ID', 'Week', 'Day', 'Goal_x', 'Muscle Group', 'Exercise Name', 'Sets', 'Reps']

display(result_df[cols_view].head(20))

,User_ID,Week,Day,Goal_x,Muscle Group,Exercise Name,Sets,Reps
572,5,0,Day 1 - Full Body A,Muscle Gain,Legs,sled 45° leg press (side pov),3,8-12
573,5,0,Day 1 - Full Body A,Muscle Gain,Chest,cable decline fly,3,8-12
574,5,0,Day 1 - Full Body A,Muscle Gain,Shoulders,dumbbell one arm upright row,3,8-12
575,5,0,Day 1 - Full Body A,Muscle Gain,Triceps,impossible dips,3,8-12
576,5,0,Day 2 - Full Body B,Muscle Gain,Back,lever front pulldown,3,8-12
577,5,0,Day 2 - Full Body B,Muscle Gain,Legs,smith leg press,3,8-12
578,5,0,Day 2 - Full Body B,Muscle Gain,Biceps,cable squatting curl,3,8-12
579,5,0,Day 2 - Full Body B,Muscle Gain,Abs,weighted side bend (on stability ball),3,8-12
580,5,1,Day 1 - Full Body A,Muscle Gain,Legs,sled 45° leg press (side pov),3,8-12
581,5,1,Day 1 - Full Body A,Muscle Gain,Chest,cable decline fly,3,8-12


In [48]:
df_final.to_csv('../../datas/dataset_final.csv', index=False)